# Pinocchio_Wrapper vs Adam_Wrapper — go2 floating-base

동일 상태 `(q, dq)`를 두 wrapper에 넣고 모든 양을 수치 비교한다. `test_core_mj_vs_pin.ipynb`와 동일한 형식.

## 비교 규약 (Pin = adam 기준)
- adam은 floating-base 입력을 `(w_H_b: 4x4, joint_q: na)` / `(base_vel: 6, joint_dq: na)`로 분리 — `Adam_Wrapper`가 내부에서 분해.
- mujoco_kernel이 dq를 (v_body, ω_body)로 변환해 넘기므로, `Adam_Wrapper`는 `BODY_FIXED_REPRESENTATION`을 dynamics 메인으로, `J_world`만 `MIXED_REPRESENTATION` 인스턴스로 처리.
- adam 결과는 torch tensor → numpy로 캐스팅해 비교.

## 알려진 한계
- adam은 **분리된 C 행렬 없음**. 다만 `coriolis_term` (= C·dq 벡터)는 제공 — `adam_wrap.coriolis`로 접근.
- `Jdot @ dq`는 adam에 직접 API 없음 → 유한차분으로 처리 (정밀도 낮음).

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[0]
sys.path.insert(0, str(_ROOT / 'mj_sim' / 'quadruped'))

import numpy as np
import torch
import mujoco
import pinocchio as pin

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Adam_Wrapper, Mujoco_Kernel

%load_ext autoreload
%autoreload 2

URDF = str(_ROOT / 'models' / 'go2' / 'urdf' / 'go2_description.urdf')
PKG  = str(_ROOT / 'models' / 'go2')
MJCF = str(_ROOT / 'models' / 'scenes' / 'go2' / 'scene.xml')

# 헬퍼: torch → numpy
def t2n(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().astype(np.float64)
    return np.asarray(x, dtype=np.float64)

## 1. 두 wrapper 인스턴스화 + 동일 상태 동기화

Mujoco_Kernel을 통해 자연스러운 (q, dq)를 만들고 두 wrapper에 모두 주입.

In [ ]:
# Pinocchio wrapper로 joint 순서 추출 (URDF tree 기준)
pin_wrap = Pinocchio_Wrapper(URDF, PKG)
joint_names_pin = [pin_wrap.model.names[i] for i in range(2, pin_wrap.model.njoints)]
print('joint_names_pin:', joint_names_pin)

# adam wrapper — 같은 joint 순서 사용. 검증은 CPU에서.
adam_wrap = Adam_Wrapper(URDF, PKG, joint_names=joint_names_pin, device='cpu', dtype=torch.float64)

# Mujoco kernel
kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names_pin)
kernel.reset_to_keyframe(0)

# 비대칭 상태 (yaw + perturb)
rng = np.random.default_rng(0)
yaw0 = np.deg2rad(30.0)
kernel.data.qpos[3] = np.cos(yaw0/2)
kernel.data.qpos[6] = np.sin(yaw0/2)
kernel.data.qpos[7:] += 0.05 * rng.standard_normal(kernel.model.nq - 7)
kernel.data.qvel[:]  = 0.1 * rng.standard_normal(kernel.model.nv)
mujoco.mj_forward(kernel.model, kernel.data)

state = FloatingBaseRobotState()
kernel.update_robot_state(state)
pin_wrap.update_model(state.q, state.dq)
adam_wrap.update_model(state.q, state.dq)

print(f'Pin  mass = {pin_wrap.mass:.6f}')
print(f'Adam mass = {adam_wrap.mass:.6f}')
print(f'nv = {pin_wrap.nv} (Pin)  / {adam_wrap.nv} (Adam)')
print(f'\nq[:7] (x,y,z,qx,qy,qz,qw) = {state.q[:7]}')

## 2. 기본 상태 — base SE3, CoM

In [ ]:
# base SE3
p_pin = pin_wrap.oMb.translation
R_pin = pin_wrap.oMb.rotation
T_adam = t2n(adam_wrap.oMb)
p_adam = T_adam[:3, 3]
R_adam = T_adam[:3, :3]
print(f'[base pos] err = {np.linalg.norm(p_pin - p_adam):.3e}')
print(f'[base rot] err = {np.linalg.norm(R_pin - R_adam):.3e}')

# CoM
com_pin  = pin_wrap.pos_com_world
vcom_pin = pin_wrap.vel_com_world
com_adam  = t2n(adam_wrap.pos_com_world)
vcom_adam = t2n(adam_wrap.vel_com_world)
print(f'\n[CoM pos] err = {np.linalg.norm(com_pin - com_adam):.3e}')
print(f'[CoM vel] err = {np.linalg.norm(vcom_pin - vcom_adam):.3e}')

## 3. Foot frame placement

Pinocchio는 `pin.SE3`, adam은 (4,4) torch tensor — 좌표 추출만 일치하면 같은 값.

In [ ]:
for key in ['FL', 'FR', 'RL', 'RR']:
    oMf_pin  = getattr(pin_wrap, f'oM_{key}')
    oMf_adam = t2n(getattr(adam_wrap, f'oM_{key}'))
    err_p = np.linalg.norm(oMf_pin.translation - oMf_adam[:3, 3])
    err_R = np.linalg.norm(oMf_pin.rotation    - oMf_adam[:3, :3])
    print(f'[{key}_foot] pos err = {err_p:.3e}, rot err = {err_R:.3e}')

## 4. 동역학 — M, nle, g

In [ ]:
M_pin   = pin_wrap.M.copy()
M_adam  = t2n(adam_wrap.M)
err_M = np.linalg.norm(M_pin - M_adam)
print(f'[M]   F-norm err = {err_M:.3e}  (||M_pin||_F = {np.linalg.norm(M_pin):.2e})')
print(f'      relative = {err_M/np.linalg.norm(M_pin)*100:.4f}%')
print(f'      symmetry pin = {np.linalg.norm(M_pin - M_pin.T):.2e}')
print(f'      symmetry adam = {np.linalg.norm(M_adam - M_adam.T):.2e}')

nle_pin  = pin_wrap.nle.copy()
nle_adam = t2n(adam_wrap.nle)
err_nle = np.linalg.norm(nle_pin - nle_adam)
print(f'\n[nle] err = {err_nle:.3e}  (||nle_pin|| = {np.linalg.norm(nle_pin):.2e})')

g_pin  = pin_wrap.g.copy()
g_adam = t2n(adam_wrap.g)
err_g = np.linalg.norm(g_pin - g_adam)
print(f'\n[g]   err = {err_g:.3e}  (||g_pin|| = {np.linalg.norm(g_pin):.2e})')

## 5. Jacobian — world & body

pinocchio LOCAL_WORLD_ALIGNED = adam MIXED_REPRESENTATION (`J_world`).
pinocchio LOCAL = adam BODY_FIXED_REPRESENTATION (`J_body`).

In [ ]:
for key in ['FL', 'FR', 'RL', 'RR']:
    Jw_pin  = pin_wrap.J_world(key)
    Jw_adam = t2n(adam_wrap.J_world(key))
    Jb_pin  = pin_wrap.J_body(key)
    Jb_adam = t2n(adam_wrap.J_body(key))
    err_w = np.linalg.norm(Jw_pin - Jw_adam)
    err_b = np.linalg.norm(Jb_pin - Jb_adam)
    print(f'[{key}] J_world err = {err_w:.3e},  J_body err = {err_b:.3e}')

# CoM Jacobian
Jcom_pin  = pin_wrap.J_com()
Jcom_adam = t2n(adam_wrap.J_com())
print(f'\n[J_com] err = {np.linalg.norm(Jcom_pin - Jcom_adam):.3e}')

## 6. Frame velocity (twist via J_world @ dq_full)

`ee_state_world`가 SE3 + (lin, ang) 반환. adam은 `J_world @ [base_vel_mixed; joint_dq]`로 합성.

In [ ]:
for key in ['FL', 'FR', 'RL', 'RR']:
    _, lin_pin, ang_pin   = pin_wrap.ee_state_world(key)
    _, lin_adam, ang_adam = adam_wrap.ee_state_world(key)
    lin_adam = t2n(lin_adam); ang_adam = t2n(ang_adam)
    err_lin = np.linalg.norm(lin_pin - lin_adam)
    err_ang = np.linalg.norm(ang_pin - ang_adam)
    print(f'[{key}] lin err = {err_lin:.3e},  ang err = {err_ang:.3e}')

## 7. Centroidal momentum — Ag, hg, Ig

In [ ]:
# Pin: hg는 pin.Force, .linear/.angular
lin_pin = np.asarray(pin_wrap.hg.linear)
ang_pin = np.asarray(pin_wrap.hg.angular)
hg_pin = np.concatenate([lin_pin, ang_pin])

# Adam: hg는 6-vector tensor (앞 3개 linear)
hg_adam = t2n(adam_wrap.hg)

print(f'[hg.linear ] err = {np.linalg.norm(lin_pin - hg_adam[:3]):.3e}')
print(f'[hg.angular] err = {np.linalg.norm(ang_pin - hg_adam[3:]):.3e}')

# Ag matrix
Ag_pin  = pin_wrap.Ag.copy()
Ag_adam = t2n(adam_wrap.Ag)
err_Ag = np.linalg.norm(Ag_pin - Ag_adam)
print(f'\n[Ag] F-norm err = {err_Ag:.3e}  (||Ag_pin||_F = {np.linalg.norm(Ag_pin):.2e})')
print(f'     relative = {err_Ag/np.linalg.norm(Ag_pin)*100:.4f}%')

# 자기검증 — 두 wrapper 모두 Ag·dq == hg 만족하는지
hg_pin_chk  = Ag_pin @ state.dq
print(f'\n[Pin self-check]  ||Ag·dq - hg|| = {np.linalg.norm(hg_pin_chk - hg_pin):.3e}')
# adam의 hg는 내부에서 Ag @ [bv_body; jdq]로 만든 거라 자명하게 0
print(f'[Adam self-check] (정의상 0)')

# Centroidal composite inertia (Ig)
Ig_pin  = pin_wrap.inertia_matrix
Ig_adam = t2n(adam_wrap.inertia_matrix)
print(f'\n[Ig] F-norm err = {np.linalg.norm(Ig_pin - Ig_adam):.3e}')
print(f'\nIg pin:\n{Ig_pin}')
print(f'\nIg adam:\n{Ig_adam}')

## 8. Coriolis 항 — adam은 C·dq 벡터만 제공

adam은 분리된 C 행렬 단독은 못 주지만, `coriolis_term`으로 **C·dq 벡터**는 제공한다 (iCub 예제 cell 22 참조).
검증: pinocchio `C @ dq` ≈ adam `coriolis`, 그리고 `nle - g` 와도 일치해야 함.

In [ ]:
# Pin: C 행렬에서 Cdq 합성
C_pin = pin_wrap.C
Cdq_pin = C_pin @ state.dq
print(f'[Pin self-check] ||C·dq + g - nle|| = {np.linalg.norm(Cdq_pin + pin_wrap.g - pin_wrap.nle):.3e}')

# Adam: coriolis_term이 직접 C·dq 벡터를 줌
Cdq_adam = t2n(adam_wrap.coriolis)
err_Cdq = np.linalg.norm(Cdq_pin - Cdq_adam)
print(f'\n[C·dq] err pin vs adam = {err_Cdq:.3e}')
print(f'       ||C·dq_pin|| = {np.linalg.norm(Cdq_pin):.3e}')

# 또 하나의 검증: nle - g == C·dq (둘 다)
print(f'\n[Adam self-check] ||nle - g - C·dq|| = '
      f'{np.linalg.norm(t2n(adam_wrap.nle) - t2n(adam_wrap.g) - Cdq_adam):.3e}')

# 분리된 C 행렬은 여전히 NotImplementedError
try:
    _ = adam_wrap.C
    print('Adam_Wrapper.C 호출 성공 (예상치 못함)')
except NotImplementedError as e:
    print(f'\nAdam_Wrapper.C → NotImplementedError (예상대로): {e}')

## 9. Jdot @ dq

adam도 직접 API (`jacobian_dot`)를 제공한다. 두 라이브러리 모두 정확한 미분 — 1e-8 이내 일치 기대.

In [ ]:
for key in ['FL', 'FR', 'RL', 'RR']:
    Jd_dq_pin  = pin_wrap.Jdot_dq_world(key)
    Jd_dq_adam = t2n(adam_wrap.Jdot_dq_world(key))
    err = np.linalg.norm(Jd_dq_pin - Jd_dq_adam)
    print(f'[{key}] err = {err:.3e}')
    print(f'       pin  = {Jd_dq_pin}')
    print(f'       adam = {Jd_dq_adam}\n')

## 10. 보조 — moment arm, world↔base 변환

In [ ]:
for key in ['FL', 'FR', 'RL', 'RR']:
    arm_pin  = pin_wrap.get_moment_arm_in_world(key)
    arm_adam = t2n(adam_wrap.get_moment_arm_in_world(key))
    print(f'[{key} moment arm] err = {np.linalg.norm(arm_pin - arm_adam):.3e}')

# world → base
p_world = np.array([1.0, 0.5, 0.3])
p_base_pin  = pin_wrap.world_to_base_frame(p_world)
p_base_adam = t2n(adam_wrap.world_to_base_frame(p_world))
print(f'\n[world_to_base] err = {np.linalg.norm(p_base_pin - p_base_adam):.3e}')

traj_world = np.random.default_rng(1).standard_normal((10, 3))
tb_pin  = pin_wrap.trajectory_world_to_base(traj_world)
tb_adam = t2n(adam_wrap.trajectory_world_to_base(traj_world))
print(f'[trajectory_world_to_base] err = {np.linalg.norm(tb_pin - tb_adam):.3e}')

## 11. R_z (yaw 추출)

In [ ]:
Rz_pin  = pin_wrap.R_z
Rz_adam = t2n(adam_wrap.R_z)
print(f'[R_z] err = {np.linalg.norm(Rz_pin - Rz_adam):.3e}')
print(f'\nRz pin:\n{Rz_pin}')
print(f'\nRz adam:\n{Rz_adam}')

## 종합 (예상)

| 양 | 예상 일치도 | 비고 |
|---|---|---|
| base SE3, CoM, foot SE3 | 1e-10 ~ 1e-12 | 같은 URDF, 같은 알고리즘 |
| `M`, `nle`, `g` | 1e-10 ~ 1e-8 | Featherstone 수치적 동일 (float64) |
| `J_world`, `J_body`, `J_com` | 1e-10 | representation 매핑만 맞으면 동일 |
| frame twist | 1e-10 | |
| `Ag`, `hg`, `Ig` | 1e-8 ~ 1e-6 | 좌표계 정의 동일 시 일치 |
| `C` matrix | — | adam 미제공 |
| `Jdot @ dq` | 1e-3 ~ 1e-5 | adam은 FD eps=1e-6, 큰 dq에서 오차 ↑ |

## 디버깅 체크리스트 (어긋날 때)
1. **joint 순서**: `joint_names_pin` ≠ adam에 넣은 순서면 M/nle 전부 망가짐
2. **base velocity representation**: mujoco_kernel이 (v_body, ω_body)로 변환해 넘기는지 재확인
3. **gravity**: adam은 6벡터, 부호 일치 확인 (`(0, 0, -9.81, 0, 0, 0)`)
4. **quat 컨벤션**: pinocchio xyzw vs adam은 회전행렬로만 받음 — `_split_state`에서 변환
5. **`Ag`/`Ig` 좌표계**: pinocchio는 world frame about CoM. adam은 representation에 따라 다를 수 있음 — 안 맞으면 representation 토글해 비교